# Version 23 — MIMO / hierarchical control of coupled zones

**Hypothesis:** Jointly selecting common AHU supply air and zone VAV actions, then coordinating a shared boiler, can improve comfort/energy trade-offs when multiple zones compete for the same equipment.

**Upper level:** Evaluate six bounded shared AHU supply-air temperature candidates. For each candidate, predict the temperature correction for *every* connected zone with a simple two-input thermal surrogate and score their **joint** tracking residual, fan/coil use, reheat and economizer cost. The same supply temperature changes all connected zones; neutral-zone spillover is explicitly penalized.

**Lower level:** Under each candidate, project VAV airflow requests onto a single AHU capacity proxy using priority-weighted water-filling. Select AHU candidates using a common hot-water scarcity price (so supply choices anticipate shared reheat demand), then project *all* VAV reheat loads onto a fixed installed-zone hot-water capacity proxy. Set boiler temperature, pressure and run-state once.

Surrogate: `ΔT_i = 0.19 · damper_i · (T_air - T_zone_i) + 3.6 · damper_i · valve_i`. Gains, AHU airflow and hot-water capacity limits are **uncalibrated normalized control proxies** (not physical kW/CFM guarantees). Six supply candidates per AHU, a 32-step airflow projection and 24 cheap scalar-price iterations keep computational cost bounded without SciPy, training or simulator access.

**Distinction from V14:** V14 applied separate airflow and water QPs *after fixing the AHU supply operating point*. V23 makes the shared supply-air decision part of a coupled-zone joint objective and recomputes the resource allocation for every candidate.

**Kaggle interface:** One policy implementation cell, final gateway cell, float32 continuous actions in [-1,1], int32 discrete actions, metadata-based dynamic topology, corrected outdoor sensor denormalization, and safe idle defaults.

**Evaluation:** This is a new unscored hypothesis. Run synthetic tests first, then compare official Kaggle score, per-zone comfort tails, energy/carbon, boiler and AHU switching, and runtime against V7/V14/V21. It does **not** guarantee a scoreboard improvement.


In [ ]:
"""Version 23: hierarchical MIMO HVAC with shared AHU and water coordination.

A bounded surrogate jointly scores shared AHU supply air and all connected
zone dampers, then coordinates reheat across the shared water plant.
This is not a calibrated plant model or a performance guarantee.
Runtime: O(AHUs * supply_candidates * zones * projection_iterations).
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


def project_shared_capacity(requested, priority, capacity):
    """Weighted Euclidean projection onto a shared nonnegative resource cap.

    Minimize sum_i 0.5 * priority_i * (x_i - requested_i)^2
    subject to 0 <= x_i <= requested_i and sum(x_i) <= capacity.
    Bisection on the common shadow price avoids SciPy and scales linearly.
    """
    requested = np.maximum(np.asarray(requested, dtype=np.float64), 0.0)
    priority = np.asarray(priority, dtype=np.float64)
    if requested.ndim != 1 or priority.shape != requested.shape:
        raise ValueError("resource vectors must have identical 1D shape")
    if (not np.isfinite(requested).all() or not np.isfinite(priority).all()
            or np.any(priority <= 0.0) or not np.isfinite(capacity)
            or capacity < 0.0):
        raise ValueError("invalid resource projection input")
    if requested.size == 0 or capacity == 0.0:
        return np.zeros_like(requested)
    if float(np.sum(requested)) <= capacity:
        return requested.copy()

    lower, upper = 0.0, float(np.max(priority * requested))
    for _ in range(32):
        price = 0.5 * (lower + upper)
        proposal = np.maximum(0.0, requested - price / priority)
        if float(np.sum(proposal)) > capacity:
            lower = price
        else:
            upper = price
    return np.maximum(0.0, requested - upper / priority)


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Hierarchical coupled-zone dispatch with explicit shared-equipment constraints.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU admits at least 30% outside air.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    # One-step, deliberately uncalibrated zone thermal surrogate.
    # delta_temperature_K = AIR_GAIN * damper * (supply_K - zone_K)
    #                       + REHEAT_GAIN * damper * reheat_valve
    AIR_GAIN = 0.19
    REHEAT_GAIN = 3.6
    COMFORT_RESPONSE = 0.52
    SUPPLY_CANDIDATES_K = (285.0, 289.0, 293.0, 297.0, 301.0, 305.0)

    # Command-space budgets, NOT measured CFM/kW capacities.
    AIR_CAPACITY_FIRST = 0.80
    AIR_CAPACITY_ADDITIONAL = 0.50
    WATER_CAPACITY_FIRST = 0.80
    WATER_CAPACITY_ADDITIONAL = 0.42

    # Surrogate objective weights; tune only against held-out buildings.
    COIL_COST = 0.022
    FAN_COST = 0.080
    REHEAT_COST = 0.22
    ECONOMIZER_COST = 0.15
    WARM_SUPPLY_TIE_BREAK = 0.00001

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        # The optional boiler switch follows all AHU switches. Never reuse
        # an AHU switch as a boiler when no hot-water plant is described.
        self.water_run = (
            self.n_discrete - 1
            if self.water_supply is not None and self.n_discrete > self.ahu_count
            else None
        )

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        # A shared boiler serves all mapped zones. Its rated capacity proxy
        # must not vary just because a different number of zones request heat.
        self.zone_count = sum(len(ahu["zones"]) for ahu in self.ahus)
        outside_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outside_mean = float(outside_stats.get("sample_mean", np.nan))
        self.outside_std = float(outside_stats.get("sample_std", np.nan))
        self.reset()

    def reset(self):
        """No episode state: decisions depend only on the current observation."""
        return None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temperature_k": temperature,
            "cooling_error_k": max(0.0, temperature - high),
            "heating_error_k": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete


    def _outdoor_temperature(self, obs, index):
        """Read outdoor Kelvin: the gateway normally standardizes this field."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outside_mean) and np.isfinite(self.outside_std)
                and self.outside_std > 0):
            kelvin = raw * self.outside_std + self.outside_mean
        else:
            kelvin = raw
        return float(kelvin) if 210.0 <= kelvin <= 330.0 else float("nan")

    def _mixed_supply(self, coil_setpoint_k, temps, outdoor_k):
        """Model AHU common supply: recirculation, mixing, cooling-only coil.

        Return (recirculation fraction, mixed air K, delivered air K). The
        economizer is used only when it can eliminate coil cooling entirely;
        its extra exhaust-fan work is penalized in the planner.
        """
        return_k = float(np.mean(temps))
        outside_k = outdoor_k if np.isfinite(outdoor_k) else return_k
        recirc = self.MAX_RETURN_FRACTION
        if (self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5
                and coil_setpoint_k < return_k):
            min_mix = self.MIN_ECONOMIZER_RECIRC * return_k + (
                1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k
            if min_mix <= coil_setpoint_k:
                recirc = self._clamp(
                    (coil_setpoint_k - outside_k) / (return_k - outside_k),
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)
        mixed_k = recirc * return_k + (1.0 - recirc) * outside_k
        return recirc, mixed_k, min(mixed_k, coil_setpoint_k)

    def _zone_demand_model(self, states):
        """Prepare desired zone corrections and priority once per AHU.

        Positive delta-T means heating; negative means cooling. The model
        never requests reheat when the building has no usable boiler.
        """
        temperature_k = np.array([s["temperature_k"] for s in states])
        cooling_error_k = np.array([s["cooling_error_k"] for s in states])
        heating_error_k = np.array([s["heating_error_k"] for s in states])
        has_damper = np.array([
            s["actions"]["damper"] is not None for s in states])
        has_reheat = np.array([
            s["actions"]["valve"] is not None for s in states])
        can_heat = has_damper & has_reheat & (self.water_run is not None)

        needs_cooling = (cooling_error_k >= self.ACTIVE_THRESHOLD_K) & has_damper
        needs_heating = (heating_error_k >= self.ACTIVE_THRESHOLD_K) & can_heat
        if not np.any(needs_cooling | needs_heating):
            return None

        desired_delta_k = np.where(
            needs_cooling,
            -np.minimum(1.8, self.COMFORT_RESPONSE * cooling_error_k),
            np.where(
                needs_heating,
                np.minimum(1.8, self.COMFORT_RESPONSE * heating_error_k),
                0.0,
            ),
        )
        worst_error_k = np.maximum(cooling_error_k, heating_error_k)
        priority = (
            1.0 + 2.2 * np.minimum(worst_error_k, 5.0)
            + 2.0 * np.maximum(worst_error_k - 5.0, 0.0) ** 2
        )
        return {
            "temperature_k": temperature_k,
            "desired_delta_k": desired_delta_k,
            "needs_cooling": needs_cooling,
            "needs_heating": needs_heating,
            "can_heat": can_heat,
            "priority": priority,
        }

    def _evaluate_supply_candidate(self, model, coil_k, outdoor_k):
        """Estimate all zone effects under *one common AHU supply setting*.

        Airflow allocation is joint: increasing one VAV consumes a fraction
        of the shared fan capacity. The supply-temperature choice is joint:
        every zone sees the same delivered air, including neutral zones.
        """
        room_k = model["temperature_k"]
        desired_delta_k = model["desired_delta_k"]
        recirc, mixed_k, supply_k = self._mixed_supply(
            coil_k, room_k, outdoor_k)

        # Cooling demand is supplied by cold air; heating demand by VAV
        # reheat. These are desired damper fractions before shared limits.
        cooling_gain = self.AIR_GAIN * np.maximum(room_k - supply_k, 0.0)
        cooling_damper = np.where(
            model["needs_cooling"] & (cooling_gain > 1e-6),
            np.clip(
                -desired_delta_k / np.maximum(cooling_gain, 1e-6),
                self.MIN_VAV_FRACTION,
                0.98,
            ),
            self.MIN_VAV_FRACTION,
        )
        reheat_gain = self.REHEAT_GAIN + self.AIR_GAIN * (supply_k - room_k)
        heating_damper = np.where(
            model["needs_heating"],
            np.clip(
                desired_delta_k / np.maximum(reheat_gain, 0.15),
                0.24,
                0.88,
            ),
            self.MIN_VAV_FRACTION,
        )
        desired_damper = np.maximum(cooling_damper, heating_damper)
        air_budget = self.AIR_CAPACITY_FIRST + (
            len(room_k) - 1) * self.AIR_CAPACITY_ADDITIONAL
        extra_air = project_shared_capacity(
            desired_damper - self.MIN_VAV_FRACTION,
            model["priority"],
            air_budget,
        )
        damper = self.MIN_VAV_FRACTION + extra_air

        # Two inputs contribute to the same zone's thermal response:
        # cold/supply air and hot-water reheat.
        air_delta_k = self.AIR_GAIN * damper * (supply_k - room_k)
        reheat_valve = np.where(
            model["can_heat"] & (desired_delta_k > 0.0),
            np.clip(
                (desired_delta_k - air_delta_k) / (self.REHEAT_GAIN * damper),
                0.0,
                1.0,
            ),
            0.0,
        )
        reheat_load = damper * reheat_valve
        predicted_delta_k = air_delta_k + self.REHEAT_GAIN * reheat_load

        # Include spillover to neutral zones instead of optimizing only the
        # most overheated or underheated VAV.
        comfort_cost = float(np.sum(
            model["priority"] * (desired_delta_k - predicted_delta_k) ** 2))
        coil_proxy = max(0.0, mixed_k - supply_k) * float(np.sum(damper))
        fan_proxy = float(np.sum(extra_air)) ** 1.5
        water_proxy = float(np.sum(reheat_load))
        exhaust_proxy = (
            self.MAX_RETURN_FRACTION - recirc) * float(np.sum(damper))
        objective = (
            comfort_cost
            + self.COIL_COST * coil_proxy
            + self.FAN_COST * fan_proxy
            + self.REHEAT_COST * water_proxy
            + self.ECONOMIZER_COST * exhaust_proxy
            + self.WARM_SUPPLY_TIE_BREAK * (self.MAX_SUPPLY_K - coil_k)
        )
        return {
            "coil_k": coil_k,
            "recirc": recirc,
            "damper": damper,
            "valve": reheat_valve,
            "objective": objective,
            "water_load": water_proxy,
            "air_budget": air_budget,
            "predicted_change": predicted_delta_k,
            "requested_change": desired_delta_k,
        }

    def _ahu_candidates(self, states, outdoor_k):
        """Compute bounded SAT alternatives once; pricing does not re-simulate."""
        if not states:
            return []
        model = self._zone_demand_model(states)
        if model is None:
            return []
        return [
            self._evaluate_supply_candidate(model, coil_k, outdoor_k)
            for coil_k in self.SUPPLY_CANDIDATES_K
        ]

    def _plan_ahu(self, states, outdoor_k):
        """Single-AHU diagnostic plan, before cross-AHU water coordination."""
        candidates = self._ahu_candidates(states, outdoor_k)
        return min(candidates, key=lambda p: p["objective"]) if candidates else None

    def _coordinate_shared_water(self, candidate_sets):
        """Select AHU operating points using one building-wide water price.

        Without scarcity, each AHU keeps its independent best plan.
        Otherwise the boiler's shared capacity imposes a marginal price on
        the predicted reheat load. This can select a warmer common SAT and
        avoid promising all zones more hot water than the plant can deliver.

        The final valve projection remains necessary: finite candidate SATs
        and the surrogate model do not guarantee physical water feasibility.
        Pricing changes *upstream choices* while the projection enforces the
        bounded *command proxy* downstream.
        """
        def select_at_price(price):
            return [
                min(options, key=lambda p: p["objective"] + price * p["water_load"])
                if options else None
                for options in candidate_sets
            ]

        def load(plans):
            return sum(p["water_load"] for p in plans if p is not None)

        selected = select_at_price(0.0)
        capacity = self._water_capacity()
        if load(selected) <= capacity:
            return selected

        # Each candidate's score is affine in price. The selected aggregate
        # reheat demand is therefore non-increasing as price increases.
        low_price, high_price = 0.0, 1.0
        for _ in range(12):
            if load(select_at_price(high_price)) <= capacity:
                break
            high_price *= 2.0

        for _ in range(24):
            mid_price = 0.5 * (low_price + high_price)
            if load(select_at_price(mid_price)) > capacity:
                low_price = mid_price
            else:
                high_price = mid_price

        selected = select_at_price(high_price)
        if load(selected) > capacity:
            # Capacity may be infeasible even at the least-water setting.
            # Keep the least-water alternatives; the valve projection
            # provides the final safety bound.
            selected = [
                min(options, key=lambda p: (p["water_load"], p["objective"]))
                if options else None
                for options in candidate_sets
            ]
        return selected

    def _dispatch_ahu(self, ahu, states, plan, continuous, discrete):
        """Write the jointly optimized AHU/VAV commands; defer boiler control."""
        if plan is None:
            return []
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(plan["coil_k"])
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(plan["recirc"])

        dampers = plan["damper"]
        if ahu["pressure"] is not None:
            # Fan pressure responds to aggregate load AND a single highly
            # open branch; all VAVs share the same fan/static pressure.
            load_proxy = max(float(np.mean(dampers)),
                             0.50 * float(np.max(dampers)))
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (load_proxy - self.MIN_VAV_FRACTION) / 0.65, 0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

        water_requests = []
        for state, damper, valve in zip(states, dampers, plan["valve"]):
            zone = state["actions"]
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
            if zone["valve"] is not None and valve > 1e-6:
                # All AHUs use the same hot-water plant. The actual valve
                # command is written after the building-wide allocation.
                water_requests.append((state, float(damper), float(valve)))
        return water_requests

    def _water_capacity(self):
        """Fixed building-wide reheat proxy, based on installed zone count."""
        return self.WATER_CAPACITY_FIRST + (
            max(0, self.zone_count - 1) * self.WATER_CAPACITY_ADDITIONAL)

    def _dispatch_water(self, requests, outdoors, continuous, discrete):
        """Project requests against one shared boiler capacity across AHUs."""
        if not requests or self.water_run is None:
            return
        demand = np.array([damper * valve for _, damper, valve in requests])
        priority = np.array([
            1.0 + 2.2 * min(state["heating_error_k"], 5.0)
            + 2.0 * max(state["heating_error_k"] - 5.0, 0.0)**2
            for state, _, _ in requests])
        budget = self._water_capacity()
        allocated = project_shared_capacity(demand, priority, budget)
        if not np.any(allocated > 1e-6):
            return
        discrete[self.water_run] = 1
        for (state, damper, _), load in zip(requests, allocated):
            continuous[state["actions"]["valve"]] = self._fraction_action(load / damper)

        coldest_error = max(
            state["heating_error_k"] for state, _, _ in requests)
        outside_k = float(np.mean(outdoors)) if outdoors else 294.0
        water_k = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside_k)
            + 5.0 * coldest_error, 320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_k - 310.0) / 40.0)
        if self.water_pressure is not None:
            load_fraction = float(np.sum(allocated)) / max(budget, 1e-6)
            pressure_fraction = self._clamp(
                0.30 + 0.63 * load_fraction, 0.35, 0.98)
            continuous[self.water_pressure] = self._fraction_action(
                pressure_fraction)

    def get_action(self, obs):
        """Hierarchical decision: zone states -> shared AHUs -> shared boiler."""
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        # Phase 1: collect candidate operating points without issuing commands.
        # The central coordinator must see every AHU's predicted water demand.
        ahu_states, candidate_sets, outdoors = [], [], []
        for ahu in self.ahus:
            states = [self._read_zone(obs, zone, fallback)
                      for zone in ahu["zones"]]
            states = [state for state in states if state is not None]
            outside_k = self._outdoor_temperature(obs, ahu["outdoor"])
            if np.isfinite(outside_k):
                outdoors.append(outside_k)
            ahu_states.append(states)
            candidate_sets.append(self._ahu_candidates(states, outside_k))

        # Phase 2: price the shared boiler before committing AHU SAT choices.
        plans = self._coordinate_shared_water(candidate_sets)

        # Phase 3: write zone/AHU commands and enforce the shared water cap.
        all_water = []
        for ahu, states, plan in zip(self.ahus, ahu_states, plans):
            all_water.extend(self._dispatch_ahu(
                ahu, states, plan, continuous, discrete))
        self._dispatch_water(all_water, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# The official inference server owns all rollouts and writes submission.parquet.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
